# 8장 실습 — 수요 만들기

지금까지는 저장소에 들어 있던 수요를 그대로 썼습니다.
이번에는 경계, 도로망, 시간대 프로파일을 차례로 반영해 요청 목록을 만듭니다.
교재 8장에 대응합니다.

만든 수요는 11장의 시뮬레이션 루프에 넣어 서비스율과 대기시간을 비교합니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 시뮬레이터가 받는 형식 (교재 8.4)

필수 컬럼은 다섯 개입니다. `id`와 `mode` 같은 컬럼을 추가할 수 있지만, 다음 다섯 값은 모든 요청에 있어야 합니다.

In [ ]:
from smartmob.data import DEMAND_COLUMNS, load_demand, validate_demand

print("필수 컬럼:", DEMAND_COLUMNS)

demand = load_demand("hanam")
print(demand.shape)
demand.head()

`validate_demand` 가 형식을 검사합니다.
좌표를 (경도, 위도) 순으로 넣는 실수를 여기서 잡습니다.

In [ ]:
validate_demand(demand)
print("[v] 형식 통과")

## 2. 시간대 패턴 (교재 8.2)

In [ ]:
import matplotlib.pyplot as plt

hourly = demand["request_time"].floordiv(60).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(hourly.index, hourly.values, color="#4C6EF5")
ax.set_xlabel("시각 (시)")
ax.set_ylabel("호출 수")
ax.set_title("하남 택시 수요의 시간대 분포")
plt.tight_layout();

## 3. 1단계 — 경계 안에 균등하게 (교재 8.5)

가장 단순한 방법입니다. 시군구 경계 안에 점을 고르게 뿌립니다.

In [ ]:
from smartmob.data import load_sigungu
from smartmob.teaching.demand_gen import generate_demand

boundary = load_sigungu("하남시")
flat = generate_demand(boundary=boundary, n=1000, seed=42, hourly=None)

validate_demand(flat)
print(flat.shape)
flat.head(3)

균등 표집은 토지 이용과 도로 접근성을 반영하지 않으므로 하천이나 산지에도 호출 위치를 만듭니다.

## 4. 2단계 — 도로 위에 (교재 8.6)

자동차 도로망의 엣지 위에서 점을 뽑습니다. 각 엣지는 직선 길이에 비례하는 확률로 선택하므로 도로 총연장이 긴 구역에 점이 더 많이 생깁니다.

In [ ]:
from smartmob.data import load_road_graph

G = load_road_graph("hanam", modes=("drive",))
on_road = generate_demand(graph=G, n=1000, seed=42, hourly=None)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharex=True, sharey=True)
for ax, df, title in [(axes[0], flat, "경계 안 균등"), (axes[1], on_road, "도로 위")]:
    ax.scatter(df["origin_lon"], df["origin_lat"], s=4, alpha=0.4, color="#4C6EF5")
    ax.set_title(title)
    ax.set_xlabel("경도")
axes[0].set_ylabel("위도")
plt.tight_layout();

## 5. 3단계 — 시간대 프로파일 (교재 8.7)

지금까지는 저녁 시간에 호출이 고르게 흩어져 있었습니다.
실제 수요는 특정 시각에 몰립니다. `hourly` 로 시간대 비중을 줍니다.

In [ ]:
from smartmob.teaching.demand_gen import HANAM_HOURLY

realistic = generate_demand(graph=G, n=1000, seed=42, hourly=HANAM_HOURLY)

fig, ax = plt.subplots(figsize=(8, 3.5))
for df, label in [(on_road, "고르게"), (realistic, "실제 프로파일")]:
    counts = df["request_time"].floordiv(60).value_counts().sort_index()
    ax.plot(counts.index, counts.values, marker="o", label=label)
ax.set_xlabel("시각 (시)")
ax.set_ylabel("호출 수")
ax.legend()
plt.tight_layout();

## 6. 만든 수요로 시뮬레이션 돌리기

시간대만 다르게 생성한 두 수요를 같은 차량과 시뮬레이션 조건으로 비교합니다.
11장에서 다룰 로컬 시뮬레이션 함수를 사용합니다.

In [ ]:
from smartmob.data import load_vehicles
from smartmob.teaching.simloop import simulate

vehicles = load_vehicles("hanam")
print(f"차량 {len(vehicles)}대")

runs = {}
for label, df in [("고르게 흩뿌린 수요", on_road), ("실제 프로파일 수요", realistic)]:
    runs[label] = simulate(df, vehicles, 1080, 1440)

import pandas as pd

pd.DataFrame({
    label: {
        "서비스율": round(r.summary()["service_rate"], 3),
        "평균대기_분": round(r.summary()["avg_waiting_time_min"], 2),
        "최대대기_분": round(r.summary()["max_waiting_time_min"], 1),
        "가동률": round(r.summary()["utilization"], 3),
    }
    for label, r in runs.items()
}).T

두 실험은 차량 80대와 요청 1,000건을 공통으로 사용합니다. 균등 시간 수요의 서비스율은 100.0%, 원자료 프로파일을 적용한 수요는 93.6%입니다. 이 예에서는 시간대 분포만 바꿔도 서비스율과 대기시간이 달라집니다.

## 7. 수요 조건 바꾸기

### 7.1 나만의 시간대 프로파일

24개짜리 리스트를 만들어 넣습니다. 합이 1이 아니어도 됩니다. 안에서 정규화합니다.
예를 들어 21시의 비중이 가장 큰 프로파일을 만들어 결과를 확인합니다.

In [ ]:
my_hourly = None      # 길이 24의 리스트. 예: [0]*18 + [1, 5, 2, 1] + [0, 0]

banner("빈칸 7.1")
if my_hourly:
    peaky = generate_demand(graph=G, n=1000, seed=42, hourly=tuple(my_hourly))
    run = simulate(peaky, vehicles, 1080, 1440)
    s = run.summary()
    print(f"[v] 서비스율 {s['service_rate']:.1%}, "
          f"평균대기 {s['avg_waiting_time_min']:.2f}분, "
          f"최대대기 {s['max_waiting_time_min']:.1f}분")
else:
    print("[ ] my_hourly 를 채우세요")

### 7.2 서비스율이 90% 미만이 되는 요청 건수

호출 건수를 1,000건에서 늘려 가며 서비스율이 90% 아래로 떨어지는 지점을 찾습니다.
차량은 80대 그대로 둡니다.

In [ ]:
break_point = None      # 서비스율이 90% 아래로 내려가는 호출 건수

# 실험용 코드입니다. 값을 바꿔 가며 돌려 보세요.
for n in [1000, 2000, 3000]:
    df = generate_demand(graph=G, n=n, seed=42, hourly=HANAM_HOURLY)
    s = simulate(df, vehicles, 1080, 1440).summary()
    print(f"호출 {n:5,}건 → 서비스율 {s['service_rate']:.1%}, "
          f"평균대기 {s['avg_waiting_time_min']:.2f}분")

banner("빈칸 7.2")
todo("서비스율이 90% 아래로 내려가는 호출 건수", break_point)

### 7.3 seed 를 바꾸면

`seed` 만 바꿔 다섯 번 돌려 서비스율의 표준편차를 구합니다.
이 실습에서 시뮬레이션 루프는 결정론적으로 실행되고, 수요 생성 과정에서 난수를 사용합니다.

In [ ]:
service_rate_std = None     # seed 5개에서 나온 서비스율의 표준편차

banner("빈칸 7.3")
todo("서비스율 표준편차", service_rate_std, fmt=lambda v: f"{v:.4f}")

## 정리

- 수요는 컬럼 다섯 개짜리 표입니다. `validate_demand` 로 형식을 먼저 확인합니다
- 경계 안 균등 표집에 도로 위치와 시간대 프로파일을 차례로 반영할 수 있습니다
- 이 실습에서는 총 요청 건수가 같아도 시간대 분포에 따라 서비스율이 달라졌습니다
- 시나리오를 비교할 때 같은 `seed`를 쓰면 표집 차이가 결과에 섞이는 것을 줄일 수 있습니다
- 9장 실습에서는 배차에 필요한 도착 예상시간을 모델로 예측합니다